# 59장 · 대화 문맥의 최근 턴과 요약 메모리

최근 턴을 자르는 방법은 오래된 발화를 제외하고, 요약은 여러 발화를 짧게 바꿉니다. 두 방법에서 어떤 사실이 사라지는지 비교합니다.

발표 장소가 회의실 B라고 기록된 대화에서 최근 두 발화만 선택해 모델에 보낼 문맥을 구성합니다. 전체 이력, 최근 문맥, 요약 중 어떤 자료에서 장소를 복원할 수 있는지 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 58장의 결과 열기

같은 폴더에 내려받은 노트북을 여세요. 먼저 my-course/agent-58.json를 읽습니다. 파일이 없으면 시작 셀의 완성 예시로 연습할 수 있습니다.

In [ ]:
from pathlib import Path
import json
folder = Path.cwd()
if (folder.name == "daily" or (folder.name.startswith("chapter") and folder.name[7:].isdigit())):
    folder = folder.parent
work = folder / "my-course"
work.mkdir(exist_ok=True)
source_file = work / "agent-58.json"
if source_file.exists():
    previous = json.loads(source_file.read_text(encoding="utf-8"))
else:
    previous = {'record': {'owner': 'alice', 'id': 'loan-01', 'text': '대출 기간은 14일입니다.'},
     'question': '대출 기간은 며칠인가요?',
     'instruction': 'lookup_loan 도구를 최대 1회 호출하세요. 빈 결과면 문서 번호를 다시 물으세요.'}
print("내 파일" if source_file.exists() else "완성 예시", previous["record"]["text"])

완성 예시 대출 기간은 14일입니다.


## 슬라이싱으로 최근 발화 선택

한 턴은 한 질문과 그 질문에 대한 응답을 묶은 단위입니다. 여기서는 각 턴의 사용자 발화만 발췌했습니다. 실제 도구를 썼다면 도구 요청과 결과도 같은 턴에 묶어야 합니다.

**대화에서 유지할 최근 턴**　최근 두 발화에는 기한과 분량이 남고 장소와 언어는 빠집니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter59/index.html#practice-space)


### 셀 01 · 리스트 슬라이싱으로 최근 발화 선택

사용자 발화 네 개 중 최근 두 개를 잘라 새 목록에 담습니다.


In [1]:
turns = ["발표 장소는 회의실 B입니다.", "발표 언어는 한국어입니다.",
         "최종 제출 기한은 금요일 18시입니다.", "슬라이드는 5장으로 준비합니다."]
recent = turns[-2:]
print(recent)

['최종 제출 기한은 금요일 18시입니다.', '슬라이드는 5장으로 준비합니다.']


기한과 분량만 남습니다. 음수 인덱스로 만든 recent는 모델에 넘길 부분을 선택한 값입니다. 원래 turns에는 네 문장이 그대로 있습니다.

최근 두 발화만 전달한 결과에는 제출 기한과 슬라이드 수가 남고 발표 장소는 빠졌습니다. 장소를 다시 물었을 때 모델이 대답하지 못한다면 먼저 입력에 회의실 B가 남아 있는지 확인합니다. 저장된 이력과 이번 호출에 포함한 문맥은 다를 수 있습니다.


#### 최근 두 발화에서 사라진 단서

<mark>최근 기록의 길이를 줄이면 앞서 확정한 장소가 문맥 밖으로 밀릴 수 있습니다.</mark> 마지막 두 발화만으로 발표 장소에 답할 수 있는지 보세요. 요약에 장소를 남겼다면 잃어버린 원문이 무엇인지 따로 확인해야 잘못된 요약을 계속 쓰지 않습니다.

선수 개념 · 35장 메시지 상태와 대화별 체크포인트

### 마지막 두 발화가 남기는 단서

<p>turns[-2:]에는 제출 기한과 슬라이드 수가 들어갑니다. 이 목록만 읽고 발표 장소를 물으면 회의실 B라는 단서를 찾을 수 없습니다.</p>


### 셀 02 · 저장된 이력에서 사실 복원

조금 전 만든 목록 대신 보관한 첫 발화를 읽습니다.


In [2]:
print(turns[0])

발표 장소는 회의실 B입니다.


‘회의실 B’가 보입니다. 기록을 잃어버린 경우와 모델에 일부만 보낸 경우를 같은 장애로 기록하면 고칠 곳을 찾기 어렵습니다.


### 원본 이력과 선택한 입력

<p>최근 두 항목을 고른 뒤 turns[0]을 다시 읽으면 회의실 B가 남아 있습니다. 이번 입력에서 빠진 발화도 원본에는 보관됩니다.</p>


## 요약과 원문 기록의 정보 손실 대조

아래 고정 요약문은 장소와 언어만 담았습니다. 세 번째 발화의 기한을 질문받았을 때 이 요약만으로 답할 수 있는지 검사합니다.


### 셀 03 · 요약에서 누락된 사실

금요일 18시라는 조건이 이 문장에 남았는지 확인하세요.


In [3]:
summary = "발표 장소는 회의실 B이며 발표 언어는 한국어입니다."
print("금요일 18시" in summary)

False


False가 나옵니다. 요약만 가진 모델은 제출 기한을 알 수 없습니다. 사용자가 기한을 물었다면 저장된 T3를 찾아 포함하거나 다시 확인해야 합니다.


### 요약이 잃어버린 사실

<p>요약에는 회의실 B와 한국어가 남았지만 금요일 18시는 빠졌습니다. 원본에서 문자열을 찾을 수 있다는 것과 이번 입력에 그 정보가 들어 있다는 것을 나눠 확인합니다.</p>


### 과제 04 · 요약과 최근 발화의 문맥 구성

요약 한 줄과 최근 발화 두 줄을 차례로 묶고 연결된 모델 입력을 줄 단위로 출력하세요.

여기서 기한은 세 번째 발화에 들어 있습니다. recent를 마지막 발화 하나로 좁히면 같은 요약문을 사용해도 입력에 기한이 남지 않습니다.


In [4]:
# summary와 recent를 묶어 줄바꿈으로 연결하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>context = [summary, *recent]
print(&#x27;\n&#x27;.join(context))</code></pre>
<p>실행 결과</p><pre><code>발표 장소는 회의실 B이며 발표 언어는 한국어입니다.
최종 제출 기한은 금요일 18시입니다.
슬라이드는 5장으로 준비합니다.</code></pre>
<p>요약이 장소와 언어를, 최근 두 문장이 기한과 분량을 제공합니다. 서비스에서는 현재 질문과 시스템 지시에 들어갈 자리도 예산에 포함합니다.</p>
</details>


### 요약과 최근 발화를 함께 쓰기

<p>장소·언어는 요약에서, 기한·슬라이드 수는 recent에서 확인할 수 있습니다. 두 입력을 이어 붙인 뒤 네 가지 질문에 필요한 단서가 실제로 있는지 찾아보세요.</p>


## 복습과 실습 점검

### 과제 05 · 장소가 바뀌었을 때 요약 갱신하기

사용자가 발표 장소를 강의실 C로 바꿨습니다. 오래된 요약의 장소만 고치고, 제출 기한과 슬라이드 수를 담은 최근 발화를 그대로 이어 붙이세요.


In [ ]:
# summary의 장소를 갱신하고 recent로부터 기한과 분량을 덧붙이세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>updated_summary = summary.replace(&quot;회의실 B&quot;, &quot;강의실 C&quot;)
print(&quot;\n&quot;.join([updated_summary, *recent]))</code></pre>
<p>실행 결과</p><pre><code>발표 장소는 강의실 C이며 발표 언어는 한국어입니다.
최종 제출 기한은 금요일 18시입니다.
슬라이드는 5장으로 준비합니다.</code></pre>
<p>첫 줄만 바뀝니다. 예전 요약을 그대로 두고 새 장소만 뒤에 덧붙이면 모델 입력 안에 발표 장소가 두 곳이 됩니다. 여기서는 고정된 문장의 장소만 치환했으므로 표현이 다른 대화의 자동 요약까지 해결하지는 않습니다.</p>
</details>


## 체크포인트에 보관한 기록과 모델에 보낼 대화 창 나누기

발표 장소를 처음에 알려 준 뒤 마감과 분량을 추가한 상황입니다. 마지막 두 메시지만 보내면 처음 장소가 빠집니다. 35장의 체크포인터에 전체 기록을 남겨 두고, 이번 요청에 건넬 문자열만 작게 만들어 봅니다.

### 셀 06 · 같은 스레드에서 전체 할 일 복원하기

수업용 대화의 각 문장을 HumanMessage로 저장합니다. 기록을 보관하는 노드는 모델을 호출하지 않습니다. 체크포인트에 어떤 메시지가 보존되는지와 최근 두 메시지에 무엇이 보이는지 확인하세요.

```mermaid
flowchart TD
    S([START]) --> K["keep · 메시지 유지"]
    K --> E([END])
```

In [5]:
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END

history_builder = StateGraph(MessagesState)
history_builder.add_node("keep", lambda state: {})
history_builder.add_edge(START, "keep")
history_builder.add_edge("keep", END)
history_app = history_builder.compile(checkpointer=InMemorySaver())
room_config = {"configurable": {"thread_id": "presentation-room"}}
history_app.invoke({"messages": [HumanMessage(content=t) for t in turns]}, room_config)
recorded = history_app.get_state(room_config).values["messages"]
print(len(recorded), [m.content for m in recorded[-2:]])

4 ['최종 제출 기한은 금요일 18시입니다.', '슬라이드는 5장으로 준비합니다.']


저장된 메시지는 4개인데 최근 두 개에는 금요일 18시와 슬라이드 5장만 있습니다. 여기서 기록을 지우지는 않았습니다. 모델 입력을 만들 때 어디까지 잘라낼지가 별도의 결정입니다.

### 과제 07 · 요약과 최신 메시지를 합쳐 한 번의 입력 만들기

제공된 summary에는 장소와 언어가 있습니다. 기록 전체를 보내지 않고 요약과 마지막 두 메시지만 모델에 보낼 문자열로 조립하세요. 생성한 문자열과 원래 체크포인트의 메시지 개수를 각각 확인하세요.

In [6]:
# summary와 recorded의 마지막 두 메시지 본문을 줄바꿈으로 연결하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def context_for_model(summary_text, messages):
    recent_lines = [m.content for m in messages[-2:]]
    return &quot;\n&quot;.join([summary_text, *recent_lines])

print(context_for_model(summary, recorded))
print(len(history_app.get_state(room_config).values[&quot;messages&quot;]))</code></pre>
<p>실행 결과</p><pre><code>발표 장소는 회의실 B이며 발표 언어는 한국어입니다.
최종 제출 기한은 금요일 18시입니다.
슬라이드는 5장으로 준비합니다.
4</code></pre>
<p>입력에는 회의실 B와 한국어, 금요일 18시와 5장이 함께 들어갑니다. 다시 읽은 메시지 개수는 여전히 4개입니다. 이 결과는 입력 범위를 시험한 것이며, 모델이 그 사실을 지켜 답하는지는 별도 질문을 보내 확인해야 합니다.</p>
</details>

<mark>모델에 보내는 대화를 줄여도 저장된 대화는 그대로 둘 수 있습니다.</mark> 체크포인트에 남은 사실도 입력 창과 요약에서 모두 빠지면 이번 호출의 모델은 그 사실을 읽을 수 없습니다.

다른 대화에서도 써야 하는 사용자 선호는 60장에서 별도의 기억 저장소로 옮깁니다.

## 대화 이력에서 다음 질문에 다시 쓸 원문 고르기

58장에서 선택한 지침을 읽고 고객에게 대출 규정을 알려 줍니다. 이어서 고객이 짧게 답해 달라고 할 때 어떤 기록이 필요한지 살펴보세요.

In [ ]:
record = previous["record"]
turns_desk = [
    {"role": "user", "content": previous["question"]},
    {"role": "assistant", "content": record["text"]},
    {"role": "user", "content": "앞으로는 짧게 답해 주세요."},
]
recent = turns_desk[-2:]
print([turn["content"] for turn in recent])

['대출 기간은 14일입니다.', '앞으로는 짧게 답해 주세요.']


마지막 두 턴에는 도구에서 읽은 원문과 고객의 새 요청이 남습니다. 첫 질문은 recent의 범위 밖에 있지만 원래 turns_desk에는 그대로 있습니다.

### 직접 확인하기

마지막 턴과 조회한 규정을 분리해 새 대화에 쓸 선호를 만드세요. 현재 답변에만 필요한 규정이 선호에 섞이면 안 됩니다.

In [ ]:
# 사용자의 마지막 메시지에서 짧은 답 요청을 찾아 short 또는 standard를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary><pre><code>preference = &quot;short&quot; if &quot;짧게&quot; in recent[-1][&quot;content&quot;] else &quot;standard&quot;
print(preference)</code></pre><p>기본 자료로 실행한 결과</p><pre><code>short</code></pre><p>사용자의 마지막 발화에서만 선호를 고릅니다. 다음 장은 그 값을 사용자별로 보관합니다.</p></details>

### 60장에 넘길 결과 저장하기

60장은 마지막 요청을 사용자 계정에 저장하고 다른 계정에서도 보이는지 검사합니다. 저장 셀을 다시 실행하면 같은 파일에 이번 결과를 기록합니다.

In [ ]:
product = {"record": record, "owner": record["owner"], "turns": recent}
target = work / "agent-59.json"
target.write_text(json.dumps(product, ensure_ascii=False, indent=2), encoding="utf-8")
print(target.name)

agent-59.json
